# Feature Engineering & Analysis

This notebook processes the raw RTIG data into engineered features and analyzes their distributions.

In [1]:
import sys
from pathlib import Path

import geopandas as gpd
import pandas as pd
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Setup
ROOT = Path('../').resolve()
sys.path.append(str(ROOT))

from src.data.preprocess import run_preprocessing

DATA_RAW = ROOT / 'data' / 'raw' / 'carreteras_RTIG.geojson'
DATA_PROCESSED = ROOT / 'data' / 'processed'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print('Running preprocessing pipeline...')
gdf, pdf = run_preprocessing(str(DATA_RAW), str(DATA_PROCESSED))
print('\nPreprocessing complete!')

Running preprocessing pipeline...
Loading data from data/raw/carreteras_RTIG.geojson...
Loaded 1602 valid features

Engineering features...

Standardizing data...

Saving processed data...
Saved GeoDataFrame: data/processed/roads_processed_gdf.parquet
Saved cleaned data: data/processed/roads_processed.parquet
Saved CSV: data/processed/roads_processed.csv


src/data/preprocess.py:81: UserWarning: Geometry is in a geographic CRS. Results from 'length' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf['length_km'] = gdf.geometry.length / 1000  # Convert from m to km


Saved GeoJSON: data/processed/roads_processed.geojson

Converting to Polars...
Polars DataFrame shape: (1602, 22)
Polars version available at data/processed/roads_processed.parquet

Preprocessing complete!


## Engineered Features Summary

In [3]:
print('Engineered features added:')
new_features = ['length_km', 'num_vertices', 'curve_complexity', 'is_tent', 'center_lat', 'center_lon']
print(gdf[new_features].describe().round(3))

Engineered features added:
       length_km  num_vertices  curve_complexity   is_tent   center_lat  \
count   1602.000      1602.000          1602.000  1602.000     1602.000   
mean      24.050      1094.435            45.014     0.334  4951663.993   
std       38.865      2837.640            52.419     0.472   335482.851   
min        0.000         2.000             3.111     0.000  3238097.399   
25%        2.206        59.250            16.448     0.000  4705206.101   
50%        6.747       218.000            23.906     0.000  5037872.458   
75%       25.617       822.500            42.070     1.000  5217725.770   
max      303.813     31256.000           377.333     1.000  5406114.494   

        center_lon  
count     1602.000  
mean   -382707.993  
std     337919.825  
min   -1851556.157  
25%    -632091.471  
50%    -401719.591  
75%    -113625.787  
max     351911.725  


## Road Length Distribution

In [4]:
# Histogram of road lengths
fig = px.histogram(
    gdf,
    x='length_km',
    nbins=50,
    title='Distribution of Road Segment Lengths',
    labels={'length_km': 'Length (km)'},
    marginal='box'
)
fig.update_xaxes(type='log')
fig.show()

print(f'Length stats:')
print(f'  Min: {gdf["length_km"].min():.3f} km')
print(f'  Max: {gdf["length_km"].max():.3f} km')
print(f'  Mean: {gdf["length_km"].mean():.3f} km')
print(f'  Total: {gdf["length_km"].sum():.1f} km')

Length stats:
  Min: 0.000 km
  Max: 303.813 km
  Mean: 24.050 km
  Total: 38528.7 km


## Curve Complexity Analysis

In [5]:
# Scatter: length vs complexity
fig = px.scatter(
    gdf,
    x='length_km',
    y='curve_complexity',
    opacity=0.6,
    title='Road Length vs Curve Complexity',
    labels={'length_km': 'Length (km)', 'curve_complexity': 'Vertices/km'}
)
fig.update_xaxes(type='log')
fig.show()

print('Curve complexity percentiles:')
print(gdf['curve_complexity'].quantile([0, 0.25, 0.5, 0.75, 0.9, 1.0]).round(2))

Curve complexity percentiles:
0.00      3.11
0.25     16.45
0.50     23.91
0.75     42.07
0.90    149.05
1.00    377.33
Name: curve_complexity, dtype: float64


## TEN-T Corridor Status

In [6]:
tent_counts = gdf['is_tent'].value_counts()
print('TEN-T status distribution:')
print(f'  TEN-T corridors: {tent_counts.get(1, 0)} ({tent_counts.get(1, 0) / len(gdf) * 100:.1f}%)')
print(f'  Non-TEN-T: {tent_counts.get(0, 0)} ({tent_counts.get(0, 0) / len(gdf) * 100:.1f}%)')

# Fix: ensure data  has both categories  
if len(tent_counts) > 1:
    labels = ['Regular Road', 'TEN-T Corridor']
    values = [tent_counts[0], tent_counts[1]]
elif tent_counts.index[0] == 1:
    # All are TEN-T
    labels = ['TEN-T Corridor']
    values = [tent_counts[1]]
else:
    # All are regular
    labels = ['Regular Road']
    values = [tent_counts[0]]

fig = px.pie(
    values=values,
    names=labels,
    title='Road Classification (TEN-T vs Regular)',
)
fig.show()

TEN-T status distribution:
  TEN-T corridors: 535 (33.4%)
  Non-TEN-T: 1067 (66.6%)


## Polars Analysis - Efficient Aggregation

In [7]:
# Using Polars for fast aggregations
print('Fast stats using Polars:')

# Group by TEN-T status
stats_by_tent = pdf.group_by('is_tent').agg([
    pl.col('length_km').sum().alias('total_km'),
    pl.col('length_km').mean().alias('avg_length_km'),
    pl.col('curve_complexity').mean().alias('avg_complexity'),
    pl.col('num_vertices').mean().alias('avg_vertices'),
])

print(stats_by_tent)

Fast stats using Polars:
shape: (2, 5)
┌─────────┬──────────────┬───────────────┬────────────────┬──────────────┐
│ is_tent ┆ total_km     ┆ avg_length_km ┆ avg_complexity ┆ avg_vertices │
│ ---     ┆ ---          ┆ ---           ┆ ---            ┆ ---          │
│ i64     ┆ f64          ┆ f64           ┆ f64            ┆ f64          │
╞═════════╪══════════════╪═══════════════╪════════════════╪══════════════╡
│ 0       ┆ 22066.507537 ┆ 20.680888     ┆ 42.826681      ┆ 867.856607   │
│ 1       ┆ 16462.186671 ┆ 30.770442     ┆ 49.376819      ┆ 1546.321495  │
└─────────┴──────────────┴───────────────┴────────────────┴──────────────┘


## Data Quality Report

In [8]:
print('Post-processing data quality:')
print(f'  Total rows: {len(gdf)}')
print(f'  Total columns (incl. geometry): {len(gdf.columns)}')

# Null analysis on processed data
null_summary = gdf.isnull().sum()
nulls_exist = null_summary[null_summary > 0]

if len(nulls_exist) > 0:
    print(f'\n  Fields with nulls:')
    print(nulls_exist)
else:
    print('  No null values found!')

print(f'\n  Geometry validity: {gdf.geometry.is_valid.sum()} / {len(gdf)}')

Post-processing data quality:
  Total rows: 1602
  Total columns (incl. geometry): 23

  Fields with nulls:
tipo_de_via          11
valido_hasta       1602
titular              10
tent_red_basica    1067
tent_corredor      1427
dtype: int64

  Geometry validity: 1602 / 1602


## Create Risk Score (Demo Scoring Model)

In [9]:
# Create a simple risk scoring model
# Higher score = higher priority for monitoring/investment

gdf['risk_score'] = 0.0

# Factor 1: Longer roads are more critical (normalized)
length_norm = (gdf['length_km'] - gdf['length_km'].min()) / (gdf['length_km'].max() - gdf['length_km'].min())
gdf['risk_score'] += length_norm * 0.3

# Factor 2: High complexity = harder to maintain/monitor
complexity_norm = (gdf['curve_complexity'] - gdf['curve_complexity'].min()) / (gdf['curve_complexity'].max() - gdf['curve_complexity'].min())
gdf['risk_score'] += complexity_norm * 0.2

# Factor 3: TEN-T corridors are more critical
gdf['risk_score'] += gdf['is_tent'] * 0.4

# Normalize to 0-100
gdf['risk_score'] = gdf['risk_score'] / gdf['risk_score'].max() * 100

print('Risk Score Distribution:')
print(gdf['risk_score'].describe().round(2))

# Visualize
fig = px.histogram(
    gdf,
    x='risk_score',
    nbins=30,
    title='Road Segment Risk Score Distribution (0-100)',
    labels={'risk_score': 'Risk Score'},
    marginal='box'
)
fig.show()

Risk Score Distribution:
count    1602.00
mean       26.74
std        29.86
min         0.08
25%         2.45
50%         8.14
75%        61.90
max       100.00
Name: risk_score, dtype: float64


## Save Enhanced Dataset

In [10]:
# Save with risk scores
gdf_scored = gdf[[
    'length_km', 'num_vertices', 'curve_complexity', 'is_tent',
    'center_lat', 'center_lon', 'risk_score', 'geometry'
]].copy()

scored_path = DATA_PROCESSED / 'roads_scored_gdf.parquet'
gdf_scored.to_parquet(scored_path)
print(f'Saved scored dataset: {scored_path}')

# Also save smaller CSV for quick reference
gdf_scored_csv = gdf_scored.drop(columns=['geometry'])
csv_path = DATA_PROCESSED / 'roads_scored.csv'
gdf_scored_csv.to_csv(csv_path, index=False)
print(f'Saved CSV: {csv_path}')

Saved scored dataset: data/processed/roads_scored_gdf.parquet
Saved CSV: data/processed/roads_scored.csv
